# AI Loan Advisory Chatbot
## Notebook 6 – Streamlit Deployment

This notebook builds the final user interface for the AI Loan Advisory Chatbot.

The application loads the previously generated artifacts, initializes the RAG pipeline, predicts user intent using the trained GRU model, retrieves relevant banking information using FAISS, generates responses through Gemini, and deploys the chatbot using Streamlit with ngrok.

This notebook does **not** retrain any models.

## Mount Google Drive

Mount Google Drive to access the saved models, embeddings, vector database, and other project files required for deployment.

In [1]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [2]:
!pip install -q streamlit
!pip install -q pyngrok
!pip install -q faiss-cpu
!pip install -q google-generativeai
!pip install -q sentence-transformers
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn
!pip install -q pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.7 MB/s eta 0:00:00


## Import Required Libraries

Import the libraries required for loading the trained models, vector database, embeddings, and building the chatbot application.

In [3]:
import os
import pickle
import warnings

import faiss
import numpy as np
import pandas as pd
import torch
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from torch import nn

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Libraries imported successfully.


## Configure Project Paths

Set the paths for all project files stored in Google Drive.


In [4]:
from pathlib import Path

# ============================================================
# Project Paths
# ============================================================

# Root directory of the project
PROJECT_DIR = Path("/content/drive/MyDrive/AI_Loan_Advisory_Chatbot")

# Important folders
PROCESSED_DIR = PROJECT_DIR / "processed"
EMBEDDING_DIR = PROCESSED_DIR / "embeddings"
VECTOR_DB_DIR = PROJECT_DIR / "vector_db"
MODEL_DIR = PROJECT_DIR / "models"

print("Project directory:", PROJECT_DIR)

Project directory: /content/drive/MyDrive/AI_Loan_Advisory_Chatbot


In [5]:
# ============================================================
# Verify Required Files
# ============================================================

required_files = {
    "Embeddings": EMBEDDING_DIR / "embeddings.pkl",
    "Metadata": EMBEDDING_DIR / "metadata.pkl",
    "FAISS Index": VECTOR_DB_DIR / "faiss_index.bin",
    "GRU Model": MODEL_DIR / "gru_intent_model.pth",
    "Tokenizer": MODEL_DIR / "tokenizer.pkl",
    "Vocabulary": MODEL_DIR / "vocabulary.pkl",
    "Label Encoder": MODEL_DIR / "label_encoder.pkl"
}

print("Checking project files...\n")

for name, path in required_files.items():
    status = "Found" if path.exists() else "Missing"
    print(f"{name:<15}: {status}")

print("\nVerification completed.")

Checking project files...

Embeddings     : Found
Metadata       : Found
FAISS Index    : Found
GRU Model      : Found
Tokenizer      : Found
Vocabulary     : Found
Label Encoder  : Found

Verification completed.


## Load Saved Artifacts

Load the saved embeddings, metadata, FAISS index, tokenizer, vocabulary, and label encoder generated in the previous notebooks.

In [6]:
# ============================================================
# Load Saved Artifacts
# ============================================================

import pickle
import faiss

# Embeddings
with open(EMBEDDING_DIR / "embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

# Metadata
with open(EMBEDDING_DIR / "metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

# FAISS Index
faiss_index = faiss.read_index(str(VECTOR_DB_DIR / "faiss_index.bin"))

# Tokenizer Configuration
with open(MODEL_DIR / "tokenizer.pkl", "rb") as f:
    tokenizer_config = pickle.load(f)

MAX_SEQUENCE_LENGTH = tokenizer_config["max_sequence_length"]
MIN_WORD_FREQUENCY = tokenizer_config["min_word_frequency"]

# Vocabulary
with open(MODEL_DIR / "vocabulary.pkl", "rb") as f:
    vocabulary = pickle.load(f)

# Label Encoder
with open(MODEL_DIR / "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [7]:
# ============================================================
# Artifact Summary
# ============================================================

print("=" * 50)
print("Loaded Project Artifacts")
print("=" * 50)

print(f"Number of Embeddings : {len(embeddings)}")
print(f"Metadata Records     : {len(metadata)}")
print(f"FAISS Vectors        : {faiss_index.ntotal}")
print(f"Vocabulary Size      : {len(vocabulary)}")
print(f"Intent Classes       : {len(label_encoder.classes_)}")

print("\nArtifacts are ready for inference.")

Loaded Project Artifacts
Number of Embeddings : 241
Metadata Records     : 241
FAISS Vectors        : 241
Vocabulary Size      : 229
Intent Classes       : 19

Artifacts are ready for inference.


## Load Embedding and Intent Models

Load the Sentence Transformer used for query embeddings and the trained GRU model for intent prediction.

In [8]:
# ============================================================
# Load Required Models
# ============================================================

from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [9]:
# ============================================================
# GRU Intent Classifier
# ============================================================

import torch
import torch.nn as nn

class GRUIntentClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        num_layers=2,
        dropout=0.3
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        embedded = self.embedding(x)

        _, hidden = self.gru(embedded)

        output = self.dropout(hidden[-1])

        return self.fc(output)

In [10]:
# ============================================================
# Load Trained GRU Model
# ============================================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

checkpoint = torch.load(
    MODEL_DIR / "gru_intent_model.pth",
    map_location=DEVICE
)

model = GRUIntentClassifier(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    output_dim=checkpoint["num_classes"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"]
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("GRU model loaded successfully.")

GRU model loaded successfully.


## Initialize Gemini

Load the Gemini API key from Google Colab Secrets and initialize the language model for response generation.

In [11]:
# ============================================================
# Initialize Gemini
# ============================================================

from google.colab import userdata
import google.generativeai as genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

gemini_model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini initialized successfully.")

Gemini initialized successfully.


## Helper Functions

Create helper functions for preprocessing queries, predicting intent, and retrieving relevant documents from the FAISS vector database.

In [12]:
# ============================================================
# Text Preprocessing
# ============================================================

import re

MAX_LENGTH = MAX_SEQUENCE_LENGTH

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-z0-9\s]", " ", text)

    tokens = text.split()

    sequence = [
        vocabulary.get(token, vocabulary.get("<UNK>", 1))
        for token in tokens
    ]

    if len(sequence) < MAX_LENGTH:
        sequence += [0] * (MAX_LENGTH - len(sequence))
    else:
        sequence = sequence[:MAX_LENGTH]

    return torch.tensor(sequence, dtype=torch.long).unsqueeze(0)

In [13]:
# ============================================================
# Intent Prediction
# ============================================================

import torch.nn.functional as F

def predict_intent(query):

    model.eval()

    inputs = preprocess_text(query).to(DEVICE)

    with torch.no_grad():

        outputs = model(inputs)

        probabilities = F.softmax(outputs, dim=1)

        confidence, prediction = torch.max(probabilities, dim=1)

    predicted_intent = label_encoder.inverse_transform(
        prediction.cpu().numpy()
    )[0]

    return predicted_intent, float(confidence)

In [14]:
# ============================================================
# Retrieve Relevant Documents
# ============================================================

def retrieve_documents(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    retrieved_docs = []

    for distance, index in zip(distances[0], indices[0]):

        if index == -1:
            continue

        doc = metadata[index].copy()
        doc["similarity"] = float(distance)

        retrieved_docs.append(doc)

    return retrieved_docs

## Build Prompt

Create the prompt that will be sent to Gemini using the retrieved banking documents as context.

In [15]:
# ============================================================
# Prompt Builder
# ============================================================

def build_prompt(query, retrieved_docs):

    context = "\n\n".join(
        doc["chunk_text"] for doc in retrieved_docs
    )

    prompt = f"""
You are an AI Loan Advisory Assistant.

Answer ONLY using the information provided in the context below.

Do not use external knowledge.

If the answer is not available in the context, respond with:

"I could not find sufficient information in the official banking documents."

--------------------------
Context
--------------------------

{context}

--------------------------
User Question
--------------------------

{query}

Provide a clear and concise answer.
"""

    return prompt

## Generate Response

Generate a response using Gemini based only on the retrieved banking information.

In [16]:
# ============================================================
# Gemini Response
# ============================================================

def generate_response(prompt):

    try:

        response = gemini_model.generate_content(prompt)

        return response.text.strip()

    except Exception as e:

        return f"Error: {str(e)}"

## RAG Pipeline

Combine intent prediction, semantic retrieval, prompt generation, and response generation into a single inference pipeline.

In [17]:
# ============================================================
# Complete RAG Pipeline
# ============================================================

SIMILARITY_THRESHOLD = 0.40


def rag_pipeline(query):

    # Predict intent
    intent, confidence = predict_intent(query)

    # Retrieve relevant documents
    retrieved_docs = retrieve_documents(query, top_k=5)

    if len(retrieved_docs) == 0:

        return {
            "answer": "I could not find any relevant information in the official banking documents.",
            "intent": intent,
            "confidence": confidence,
            "sources": []
        }

    # Check similarity
    best_similarity = retrieved_docs[0]["similarity"]

    if best_similarity < SIMILARITY_THRESHOLD:

        return {
            "answer": "The available banking documents do not contain sufficient information to answer this query.",
            "intent": intent,
            "confidence": confidence,
            "sources": retrieved_docs
        }

    # Build prompt
    prompt = build_prompt(query, retrieved_docs)

    # Generate answer
    answer = generate_response(prompt)

    return {
        "answer": answer,
        "intent": intent,
        "confidence": confidence,
        "sources": retrieved_docs
    }

In [18]:
print(metadata[0].keys())

dict_keys(['chunk_id', 'bank', 'loan_type', 'document_type', 'section', 'source_file', 'page_number', 'chunk_text', 'word_count'])


## Format Retrieved Sources

Prepare the retrieved document information for display in the chatbot interface.

In [19]:
# ============================================================
# Format Source Information
# ============================================================

def format_sources(retrieved_docs):

    sources = []

    for doc in retrieved_docs:

        sources.append({

            "Bank": doc["bank"],
            "Loan Type": doc["loan_type"],
            "Document": doc["source_file"],
            "Section": doc["section"],
            "Page": doc["page_number"],
            "Similarity": round(float(doc["similarity"]), 4)

        })

    return sources

## Complete RAG Pipeline

Combine intent prediction, document retrieval, prompt generation, response generation, and source attribution into a single inference pipeline.

In [20]:
# ============================================================
# Complete RAG Pipeline
# ============================================================

def rag_pipeline(query):

    # Predict intent
    intent, confidence = predict_intent(query)

    # Retrieve relevant documents
    retrieved_docs = retrieve_documents(query, top_k=5)

    if len(retrieved_docs) == 0:

        return {

            "answer": "I could not find relevant information in the official banking documents.",

            "intent": intent,

            "confidence": confidence,

            "sources": []

        }

    # Build Prompt
    prompt = build_prompt(query, retrieved_docs)

    # Generate Response
    answer = generate_response(prompt)

    # Format Sources
    sources = format_sources(retrieved_docs)

    return {

        "answer": answer,

        "intent": intent,

        "confidence": confidence,

        "sources": sources

    }

## Test the RAG Pipeline

Run a sample query to verify that the complete pipeline is working correctly.

In [21]:
# ============================================================
# Sample Test
# ============================================================

query = "What documents are required for a home loan?"

result = rag_pipeline(query)

print("=" * 70)
print("User Query")
print("=" * 70)
print(query)

print("\nPredicted Intent :", result["intent"])
print("Confidence       :", round(result["confidence"], 4))

print("\nAnswer")
print("-" * 70)
print(result["answer"])

print("\nRetrieved Sources")
print("-" * 70)

for source in result["sources"]:

    print(f"Bank       : {source['Bank']}")
    print(f"Loan Type  : {source['Loan Type']}")
    print(f"Document   : {source['Document']}")
    print(f"Section    : {source['Section']}")
    print(f"Page       : {source['Page']}")
    print(f"Similarity : {source['Similarity']}")
    print("-" * 70)

User Query
What documents are required for a home loan?

Predicted Intent : loan_eligibility
Confidence       : 0.5519

Answer
----------------------------------------------------------------------
For home loan approval, you need to submit the following documents for the applicant and all co-applicants along with the completed and signed Loan application form:

**1. Identity And Residence (KYC) Documents:**
*   PAN Card or Form 60.
*   Officially Valid Documents (OVDs) for establishing legal name and current address (any one): Passport, Driving license, Election/Voters identification card, Job card issued by NREGA, Letter issued by the National Population Register, Proof of possession of Aadhaar Number.

**2. Income Documents:**
*   Last 3 months' Salary Slips (for salaried).
*   Last 6 months' Bank Statements, showing salary credits (for salaried).
*   Latest Form-16 and IT returns (for salaried).
*   Income Tax Returns along with computation of income for at least the last 2 Assessm

## Generate Streamlit Application

Create the Streamlit application file that provides the user interface for the AI Loan Advisory Chatbot.

In [22]:
from pathlib import Path

STREAMLIT_DIR = PROJECT_DIR / "streamlit_app"
STREAMLIT_DIR.mkdir(exist_ok=True)

APP_FILE = STREAMLIT_DIR / "app.py"

print(APP_FILE)

/content/drive/MyDrive/AI_Loan_Advisory_Chatbot/streamlit_app/app.py


In [23]:
app_code = r'''
import streamlit as st
import pickle
import faiss
import torch
import google.generativeai as genai
from pathlib import Path
from sentence_transformers import SentenceTransformer

st.set_page_config(
    page_title="AI Loan Advisory Chatbot",
    page_icon="🏦",
    layout="wide"
)

st.title("🏦 AI Loan Advisory Chatbot")

st.markdown("""
Ask questions related to:

- SBI Loans
- HDFC Loans
- Home Loan
- Car Loan
- Personal Loan
- Education Loan
- Gold Loan
- Loan Against Property

The chatbot answers only from official banking documents.
""")

st.info(
    "Example: What documents are required for an HDFC Home Loan?"
)

if "history" not in st.session_state:
    st.session_state.history = []

query = st.chat_input("Ask your loan related question...")

if query:

    st.chat_message("user").write(query)

    with st.spinner("Searching official banking documents..."):

        result = rag_pipeline(query)

    st.chat_message("assistant").write(result["answer"])

    st.markdown("---")

    st.subheader("Predicted Intent")

    st.write(result["intent"])

    st.write(
        f"Confidence : {result['confidence']:.2%}"
    )

    st.markdown("---")

    st.subheader("Retrieved Sources")

    for source in result["sources"]:

        with st.expander(source["Document"]):

            st.write(f"**Bank:** {source['Bank']}")
            st.write(f"**Loan Type:** {source['Loan Type']}")
            st.write(f"**Section:** {source['Section']}")
            st.write(f"**Page:** {source['Page']}")
            st.write(f"**Similarity:** {source['Similarity']}")
'''

In [24]:
with open(APP_FILE, "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully.")

app.py created successfully.


## Generate Requirements File

Create the requirements.txt file required for deployment.

In [25]:
requirements = """
streamlit
torch
faiss-cpu
numpy
pandas
sentence-transformers
google-generativeai
scikit-learn
pyngrok
"""

with open(STREAMLIT_DIR / "requirements.txt", "w") as f:
    f.write(requirements.strip())

print("requirements.txt created successfully.")

requirements.txt created successfully.


In [26]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

## Launch Streamlit

Run the Streamlit application locally and expose it using ngrok.

In [51]:
from pyngrok import ngrok
from google.colab import userdata
import os

NGROK_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

ngrok.set_auth_token(NGROK_TOKEN)

print("Ngrok configured successfully.")

Ngrok configured successfully.


In [52]:
%%writefile run.sh

streamlit run /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/streamlit_app/app.py \
--server.port 8501 \
--server.headless true \
--server.enableCORS false \
--server.enableXsrfProtection false

Overwriting run.sh


In [53]:
import subprocess
import time

process = subprocess.Popen(
    ["bash", "run.sh"]
)

time.sleep(8)

public_url = ngrok.connect(8501)

print("=" * 60)
print("Streamlit Application Running")
print("=" * 60)
print(public_url)

Streamlit Application Running
NgrokTunnel: "https://lunchtime-gamma-canola.ngrok-free.dev" -> "http://localhost:8501"


In [50]:
!pkill -f streamlit

In [40]:
!streamlit run /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/streamlit_app/app.py --server.headless true --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦⠧⠇Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-07-12 07:23:30.922 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.148.105.29:8501

  Stopping...
^C
